In [1]:
import pickle
import time

from omnes_pro_uno.fphse import Fphse

OP_ADD = b"\x01"

with open("../lib/enron_kw100_00-01_timeline.pkl", "rb") as f:
    enron = pickle.load(f)
print("enron loaded")

data_owner_num = len(enron.keys())
writers = list(enron.keys())

# Setup
setup_t = time.time()

fphse = Fphse(data_owner_num)
msk = fphse.rsetup()

wk_vec = []
e_tkn_vec = []
st_vec = []
enb_vec = []
b_vec = []


def setup_data_owner(i):
    b_vec.append({})
    wk, st, enb = fphse.wsetup()
    e_tkn, st = fphse.rebuild(i, wk, b_vec[i], st)
    wk_vec.append(wk)
    e_tkn_vec.append(e_tkn)
    st_vec.append(st)
    enb_vec.append(enb)


# Data owner setup can be parallelized
setup_data_owner(0)

# Setup done
print(f"setup (s): {time.time() - setup_t}")

for i in range(1, data_owner_num):
    setup_data_owner(i)

# Update
update_t = time.time()
update_epoch_t = time.time()


def epoch_rotation():
    fphse.edsse.set_epoch(fphse.edsse.get_epoch() + 1)
    for i in range(data_owner_num):
        e_tkn_vec[i], st_vec[i] = fphse.rebuild(i, wk_vec[i], b_vec[i], st_vec[i])


epoch = 0
total_update_d = 0.0

for year in ["2000", "2001"]:
    for month in range(1, 13):
        if month < 10:
            month_s = "0" + str(month)
        else:
            month_s = str(month)
        date = year + "-" + month_s

        for i, writer in enumerate(writers):
            map = enron[writer]
            posts = map[date]
            for post in posts:
                pid = post["id"].encode()
                if len(pid) > 47:
                    pid = pid[:47]
                else:
                    pid = pid + b"\0" * (47 - len(pid))

                for kw in post["keywords"]:
                    u_no_sse, st_vec[i] = fphse.update_token(
                        i, b_vec[i], wk_vec[i], st_vec[i], OP_ADD, kw.encode(), pid
                    )
                    enb_vec[i], e_tkn_vec[i] = fphse.update(u_no_sse, enb_vec[i], e_tkn_vec[i])
        epoch_rotation()

        print(f"epoch {epoch} done (s): {time.time() - update_epoch_t}")
        update_epoch_t = time.time()
        epoch += 1

# Update done
print(f"all epoch done (s): {time.time() - update_t}")

enron loaded
setup (s): 0.22191357612609863
epoch 0 done (s): 62.257848024368286
epoch 1 done (s): 42.82745695114136
epoch 2 done (s): 46.81520700454712
epoch 3 done (s): 52.599496841430664
epoch 4 done (s): 56.92930483818054
epoch 5 done (s): 80.19766736030579
epoch 6 done (s): 74.27069091796875
epoch 7 done (s): 80.60050225257874
epoch 8 done (s): 85.60185241699219
epoch 9 done (s): 98.67150616645813
epoch 10 done (s): 118.93110084533691
epoch 11 done (s): 128.65070152282715
epoch 12 done (s): 131.83989071846008
epoch 13 done (s): 133.52800798416138
epoch 14 done (s): 139.5050938129425
epoch 15 done (s): 159.1186282634735
epoch 16 done (s): 177.29367423057556
epoch 17 done (s): 174.32229781150818
epoch 18 done (s): 177.57873106002808
epoch 19 done (s): 179.07850670814514
epoch 20 done (s): 191.94995069503784
epoch 21 done (s): 205.72719287872314
epoch 22 done (s): 198.45837664604187
epoch 23 done (s): 197.147531747818
all epoch done (s): 2993.9039001464844


In [5]:
s = [0, 1]
s_no_sse = fphse.search_token(msk, s, b'maggi')
r, _, _ = fphse.search(s_no_sse, s, enb_vec, e_tkn_vec)
print(r)

[b'<27548998.1075862038812.JavaMail.evans@thyme>\x00\x00', b'<13291553.1075862036063.JavaMail.evans@thyme>\x00\x00', b'<17370281.1075862040976.JavaMail.evans@thyme>\x00\x00', b'<29236586.1075863062759.JavaMail.evans@thyme>\x00\x00', b'<32964267.1075862031901.JavaMail.evans@thyme>\x00\x00', b'<2369675.1075862026709.JavaMail.evans@thyme>\x00\x00\x00', b'<15077406.1075862037055.JavaMail.evans@thyme>\x00\x00', b'<22926499.1075862038882.JavaMail.evans@thyme>\x00\x00', b'<5404917.1075845230703.JavaMail.evans@thyme>\x00\x00\x00', b'<3733948.1075845234390.JavaMail.evans@thyme>\x00\x00\x00', b'<28288487.1075854989693.JavaMail.evans@thyme>\x00\x00', b'<10326874.1075863062414.JavaMail.evans@thyme>\x00\x00', b'<2828017.1075845230961.JavaMail.evans@thyme>\x00\x00\x00', b'<16205987.1075862026549.JavaMail.evans@thyme>\x00\x00', b'<18531100.1075845234954.JavaMail.evans@thyme>\x00\x00', b'<25139189.1075862033333.JavaMail.evans@thyme>\x00\x00', b'<6726935.1075862033022.JavaMail.evans@thyme>\x00\x00\x00'

In [6]:
s_num_vec = [5, 10, 15, 20, 25]
s_vec = [list(range(i)) for i in s_num_vec]
for s in s_vec:
    search_token_t = time.time()
    w = b'maggi'
    s_no_sse = fphse.search_token(msk, s, w)
    print(f"search token (s): {time.time() - search_token_t}")

    search_t = time.time()
    r, _, _ = fphse.search(s_no_sse, s, enb_vec, e_tkn_vec)
    print(f"search (s): {time.time() - search_t}")

search token (s): 0.0013010501861572266
search (s): 8.15324854850769
search token (s): 0.00107574462890625
search (s): 29.738529443740845
search token (s): 0.0010704994201660156
search (s): 67.08879232406616
search token (s): 0.00115203857421875
search (s): 114.70470428466797
search token (s): 0.0010890960693359375
search (s): 177.9782314300537


In [2]:
epoch_rotation_t = time.time()
epoch_rotation()
print(f"epoch rotation (s): {time.time() - epoch_rotation_t}")

epoch rotation (s): 200.01753091812134
